# Scratch: Brisnet DRF pipeline

Experiment with `scripts/load_drf.py` and `scripts/brisnet_columns.py` against files under `data/`. Safe to edit freely; this notebook is not meant for production.

In [25]:
from __future__ import annotations

import sys
from pathlib import Path


def project_root() -> Path:
    """Resolve repo root whether the kernel cwd is the project or a subfolder."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "load_drf.py").is_file():
            return candidate
    return here


ROOT = project_root()
SCRIPTS = ROOT / "scripts"
DATA_DIR = ROOT / "data"

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import importlib.util

import pandas as pd


def _import_script_module(name: str):
    """Load ``scripts/<name>.py`` by path so Jupyter always uses the repo copy (not a stale import)."""
    path = SCRIPTS / f"{name}.py"
    if not path.is_file():
        raise FileNotFoundError(path)
    unique = f"_derby_nb_{name}"
    spec = importlib.util.spec_from_file_location(unique, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


from brisnet_columns import BRISNET_COLUMNS

_ldr = _import_script_module("load_drf")
load_drf = _ldr.load_drf
past_performances_long = _ldr.past_performances_long
add_field_relative_features = _ldr.add_field_relative_features
FIELD_RELATIVE_NUMERIC_COLUMNS = _ldr.FIELD_RELATIVE_NUMERIC_COLUMNS
FIELD_RELATIVE_GROUP_KEYS = _ldr.FIELD_RELATIVE_GROUP_KEYS
select_kentucky_derby = _ldr.select_kentucky_derby

print("ROOT:", ROOT)
print("BRISNET_COLUMNS:", len(BRISNET_COLUMNS))

ROOT: /Users/jessica.lin/Documents/Projects/Misc/Kentucky Derby/Derby 2026
BRISNET_COLUMNS: 1435


In [26]:
# Default sample file — change or add more paths as needed
DRF_PATH = DATA_DIR / "CDX0503.DRF"
assert DRF_PATH.is_file(), f"Missing {DRF_PATH}"

df = load_drf(DRF_PATH)
df.shape

(187, 1435)

In [27]:
# Kentucky Derby field — filtering logic lives in `load_drf.select_kentucky_derby`
ky_derby = select_kentucky_derby(df)
ky_derby.drop_duplicates(subset=["track", "date", "race"])[
    ["track", "date", "race", "distance", "surface", "purse", "todays_race_classification"]
]

,track,date,race,distance,surface,purse,todays_race_classification
136,CD,2025-05-03,12,2200,D,5000000,KyDerby-G1


In [28]:
# Z-scores & ranks use group FIELD_RELATIVE_GROUP_KEYS (track, date, race); same stats if you pass full df
df_feat = add_field_relative_features(ky_derby)
len(FIELD_RELATIVE_NUMERIC_COLUMNS), df_feat.shape

(82, (21, 1602))

In [29]:
df_feat

,track,date,race,post_position,entry,distance,surface,reserved_8,race_type,age_sex_restrictions,...,rank_of_work_9_field_rank,rank_of_work_10_field_z,rank_of_work_10_field_rank,rank_of_work_11_field_z,rank_of_work_11_field_rank,rank_of_work_12_field_z,rank_of_work_12_field_rank,morn_line_implied_prob,morn_line_implied_prob_field_z,morn_line_implied_prob_field_rank
136,CD,2025-05-03,12,1,NaN,2200,D,NaN,G1,BON,...,11.0,-0.380080,14.0,-0.296149,10.0,-0.855972,18.0,0.050000,-0.319781,8.0
137,CD,2025-05-03,12,2,NaN,2200,D,NaN,G1,BON,...,16.0,-0.251697,9.0,-0.134458,6.0,2.110010,1.0,0.033333,-0.543628,12.0
138,CD,2025-05-03,12,3,NaN,2200,D,NaN,G1,BON,...,5.0,-0.251697,9.0,-0.554853,15.0,0.775318,5.0,0.033333,-0.543628,12.0
139,CD,2025-05-03,12,4,NaN,2200,D,NaN,G1,BON,...,16.0,-0.380080,14.0,-0.263810,9.0,-0.806539,16.0,0.083333,0.127912,4.0
140,CD,2025-05-03,12,5,NaN,2200,D,NaN,G1,BON,...,13.0,-0.139363,6.0,-0.199134,8.0,-0.757106,14.0,0.033333,-0.543628,12.0
141,CD,2025-05-03,12,6,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.033333,-0.543628,12.0
142,CD,2025-05-03,12,7,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.066667,-0.095934,7.0
143,CD,2025-05-03,12,8,NaN,2200,D,NaN,G1,BON,...,14.0,-0.299841,11.0,-0.522515,13.0,-0.658240,12.0,0.333333,3.485614,1.0
144,CD,2025-05-03,12,9,NaN,2200,D,NaN,G1,BON,...,19.0,-0.331936,12.0,-0.748881,17.0,0.231555,6.0,0.083333,0.127912,4.0
145,CD,2025-05-03,12,10,NaN,2200,D,NaN,G1,BON,...,18.0,-0.412176,19.0,-0.425501,11.0,-0.757106,14.0,0.050000,-0.319781,8.0


In [22]:
# Entry list preview (same spirit as `python scripts/load_drf.py`)
cols = ["track", "date", "race", "post_position", "horse_name"]
df_feat[cols].head(12)

,track,date,race,post_position,horse_name
136,CD,20250503,12,1,CITIZEN BULL
137,CD,20250503,12,2,NEOEQUOS
138,CD,20250503,12,3,FINAL GAMBIT
139,CD,20250503,12,4,RODRIGUEZ
140,CD,20250503,12,5,AMERICAN PROMISE
141,CD,20250503,12,6,ADMIRE DAYTONA
142,CD,20250503,12,7,LUXOR CAFE
143,CD,20250503,12,8,JOURNALISM
144,CD,20250503,12,9,BURNHAM SQUARE
145,CD,20250503,12,10,GRANDE


In [30]:
# All columns are read as strings; cast explicitly where you need numerics
pd.to_numeric(df_feat["bris_prime_power_rating"], errors="coerce").describe()

count     19.000000
mean     143.372105
std        3.055859
min      137.050000
25%      141.910000
50%      143.780000
75%      145.530000
max      148.460000
Name: bris_prime_power_rating, dtype: float64

In [31]:
# Past-performance columns use _pp1 … _pp10 suffixes
pp_like = df.columns[df.columns.str.contains(r"_pp\d+$", regex=True)]
len(pp_like), pp_like[:8].tolist()

(989,
 ['race_date_pp1',
  'race_date_pp2',
  'race_date_pp3',
  'race_date_pp4',
  'race_date_pp5',
  'race_date_pp6',
  'race_date_pp7',
  'race_date_pp8'])

In [33]:
# Long-format past performances (can be large / slow on full files)
pp_long = past_performances_long(df)
pp_long

,track,date,race,post_position,horse_name,race_back,age_sex_restrictions,alt_comment_line,apprentice_wt_allow,bris_10f_pace_fig,...,third_place_weight,track_code,track_condition,track_variant,trainer,trip_comment,weight,winner_margin,winner_name,winner_weight
0,CD,2025-05-03,1,10,KNICKS GATE,1,BUN,NaN,NaN,NaN,...,125,KEE,FT,18,COLEBROOK BEN,Step slw;3p;imprv late,125,0.50,DR. PARK,125
1,CD,2025-05-03,1,11,COFFEE TALK,1,BON,NaN,NaN,NaN,...,122,FG,FT,16,COX BRAD H,2w;btw;bid 4w;leand in,122,0.75,PRETTY CAPABLE,122
2,CD,2025-05-03,1,11,COFFEE TALK,2,BON,NaN,NaN,NaN,...,120,FG,FT,10,COX BRAD H,2wd btw;chased;yielded,120,2.00,YINZER,120
3,CD,2025-05-03,1,12,CHILLAX,1,BON,NaN,NaN,NaN,...,120,KEE,FT,12,MOTT WILLIAM I,2p;rail bid;fended off,120,0.50,SHANGRALA ROAD,120
4,CD,2025-05-03,1,12,CHILLAX,2,BON,NaN,NaN,NaN,...,118,GP,FT,9,MOTT WILLIAM I,Hit gate;2p;empty,118,9.25,DISRUPTOR,118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1307,CD,2025-05-03,9,9,GIOCOSO,4,AON,NaN,NaN,NaN,...,118,KEE,FM,14,DESORMEAUX J KEITH,2p-ins;shft out;ran on,118,1.50,MINARET STATION,118
1308,CD,2025-05-03,9,9,GIOCOSO,5,AON,NaN,NaN,NaN,...,122,CD,FT,15,DESORMEAUX J KEITH,Tug 3w;turned away 1/4,122,2.75,JONATHAN'S WAY,122
1309,CD,2025-05-03,9,9,GIOCOSO,6,AON,NaN,NaN,NaN,...,119,ELP,FM,10,DESORMEAUX J KEITH,Prompt 2wd;kicked away,119,6.25,GIOCOSO,119
1310,CD,2025-05-03,9,9,GIOCOSO,7,AON,NaN,NaN,NaN,...,119,ELP,FM,12,DESORMEAUX J KEITH,Ins;4p1/8;surged;missd,119,0.50,SING SING,119


In [44]:
comb = pd.read_csv(DATA_DIR / "processed/combined.csv")
comb

,track,date,race,post_position,entry,distance,surface,reserved_8,race_type,age_sex_restrictions,...,reserved_1432,reserved_1433,reserved_1434,reserved_1435,_source_drf,target_FP,official_final_odds,year,target_top3,target_top5
0,CD,2021-05-01,12,1,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,xnOV7B,NaN,CDX0501-2021.DRF,8,9.9,2021,0,0
1,CD,2021-05-01,12,2,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,vr:Mad,NaN,CDX0501-2021.DRF,11,49.9,2021,0,0
2,CD,2021-05-01,12,3,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,/:<)[e,NaN,CDX0501-2021.DRF,14,43.5,2021,0,0
3,CD,2021-05-01,12,4,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,kqCe!W,NaN,CDX0501-2021.DRF,6,49.0,2021,0,0
4,CD,2021-05-01,12,5,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,0WkraZ,NaN,CDX0501-2021.DRF,10,43.4,2021,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,CD,2020-09-05,14,14,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,gPl)RN,NaN,CDX0905-2020.DRF,12,50.0,2020,0,0
163,CD,2020-09-05,14,15,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,d&yq!W,NaN,CDX0905-2020.DRF,8,12.7,2020,0,0
164,CD,2020-09-05,14,16,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,MOo_JT,NaN,CDX0905-2020.DRF,4,7.6,2020,0,1
165,CD,2020-09-05,14,17,NaN,2200,D,NaN,G1,BON,...,NaN,NaN,JMoSp1,NaN,CDX0905-2020.DRF,2,0.7,2020,1,1


In [37]:
comb.shape

(1522, 1438)

---

**Tips**

- Spec reference: [Brisnet single-file DRF fields](https://support.brisnet.com/hc/en-us/articles/360056092092).
- Drop-in another `.DRF` path in the load cell to compare cards.
- If imports fail, run the first cell after opening the notebook from this project (or fix `project_root()`).
- In `past_performances_long`, PP fields that share a name with an id column (e.g. `post_position`) are prefixed with `pp_` in the wide frame.
- `add_field_relative_features` adds `*_field_z` / `*_field_rank` by race (`track`, `date`, `race`). Implied win probability from morning line: `morn_line_implied_prob` plus its rank and z. Z uses sample `std` (Excel `STDEV` / `STDEV.S`, `ddof=1`). `weight` is ranked with rank 1 = lightest; other metrics use rank 1 = largest value.